In [374]:
import os
import glob
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

In [375]:
BASEDIR = '/home/bhkim/github/bruit-classification'
DATADIR = os.path.join(BASEDIR, 'data')
MODELDIR = os.path.join(BASEDIR, 'model')

MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
SAMPLE_RATE = 16000
BATCH_SIZE = 1

In [376]:
# for f in glob.glob(os.path.join(DATADIR, '**', '**', '*.wav')):
#     if f.endswith('_16b.wav'):
#         print(os.system(f'soxi {f}'))
#     else:
#         print(f)
#         os.system(f'sox {f} -b 16 -r {SAMPLE_RATE} {f.replace(".wav","_16b.wav")}')

In [377]:
yamnet_model = hub.load(MODEL_HANDLE)

In [378]:
@tf.function
def load_wav_and_label(audio_file, label):
    # audio_data, _ = librosa.load(audio_file, sr=SAMPLE_RATE)
    # audio_data = tf.convert_to_tensor(audio_data)
    audio = tf.io.read_file(audio_file)
    audio, sample_rate = tf.audio.decode_wav(audio)
    audio = tf.squeeze(audio, axis=-1)
    return audio, label
    

In [379]:
train_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'train/**/*_16b.wav')), dtype='object')
train_audio_label_list = [f.split('/')[-2] for f in train_audio_file_list]

val_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'val/**/*_16b.wav')), dtype='object')
val_audio_label_list = [f.split('/')[-2] for f in val_audio_file_list]

test_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'test/**/*_16b.wav')), dtype='object')
test_audio_label_list = [f.split('/')[-2] for f in test_audio_file_list]

In [380]:
class_name_dict = {}
i = 0
for label in train_audio_label_list:
    if not label in list(class_name_dict.keys()):
        class_name_dict[label] = i
        i += 1

In [381]:
train_audio_label_list = pd.Series([class_name_dict[i] for i in train_audio_label_list], dtype='int64')
val_audio_label_list = pd.Series([class_name_dict[i] for i in val_audio_label_list], dtype='int64')
test_audio_label_list = pd.Series([class_name_dict[i] for i in test_audio_label_list], dtype='int64')

In [382]:
train_ds = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list))
train_ds = train_ds.map(load_wav_and_label)

val_ds = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list))
val_ds = val_ds.map(load_wav_and_label)

test_ds = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list))
test_ds = test_ds.map(load_wav_and_label)

In [383]:
@tf.function
def extract_embedding(wav_data, label):
  scores, embeddings, spectrogram = yamnet_model(wav_data)
  num_embeddings = tf.shape(embeddings)[0]
  return embeddings, tf.repeat(label, num_embeddings)

In [384]:
train_ds = train_ds.map(extract_embedding)
val_ds = val_ds.map(extract_embedding)
test_ds = test_ds.map(extract_embedding)

In [385]:
train_ds = train_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [386]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    # tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(len(list(class_name_dict.keys())))
], name='bruit_yamnet')

model.summary()

Model: "bruit_yamnet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_20 (Dense)            (None, 5)                 5125      
                                                                 
Total params: 5,125
Trainable params: 5,125
Non-trainable params: 0
_________________________________________________________________


In [387]:
model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-4
                  ),
              metrics=['accuracy'])

callback = tf.keras.callbacks.EarlyStopping(monitor='loss',
                                            patience=3,
                                            restore_best_weights=True)

In [388]:
history = model.fit(train_ds,
                    epochs=80,
                    validation_data=val_ds,
                    callbacks=callback)

Epoch 1/80


124/145 [========================>.....] - ETA: 0s - loss: 1.5980 - accuracy: 0.2772       WARNING:tensorflow:Model was constructed with shape (None, 1024) for input KerasTensor(type_spec=TensorSpec(shape=(None, 1024), dtype=tf.float32, name='yamnet_embedding'), name='yamnet_embedding', description="created by layer 'yamnet_embedding'"), but it was called on an input with incompatible shape (None, None, 1024).


145/145 [==============================] - 7s 15ms/step - loss: 1.6206 - accuracy: 0.2721 - val_loss: 1.5453 - val_accuracy: 0.2967
Epoch 2/80
145/145 [==============================] - 0s 972us/step - loss: 1.5288 - accuracy: 0.3430 - val_loss: 1.5108 - val_accuracy: 0.3386
Epoch 3/80
145/145 [==============================] - 0s 930us/step - loss: 1.4800 - accuracy: 0.3876 - val_loss: 1.4790 - val_accuracy: 0.3508
Epoch 4/80
145/145 [==============================] - 0s 974us/step - loss: 1.4369 - accuracy: 0.4119 - val_loss: 1.4474 - val_accuracy: 0.3647
Epoch 5/80
145/145 [==============================] - 0s 1ms/step - loss: 1.4082 - accuracy: 0.4362 - val_loss: 1.4219 - val_accuracy: 0.3822
Epoch 6/80
145/145 [==============================] - 0s 1ms/step - loss: 1.3761 - accuracy: 0.4429 - val_loss: 1.3964 - val_accuracy: 0.3822
Epoch 7/80
145/145 [==============================] - 0s 965us/step - loss: 1.3495 - accuracy: 0.4497 - val_loss: 1.3754 - val_accuracy: 0.3857
Epoch 8/

In [389]:
loss, accuracy = model.evaluate(test_ds)
print(loss)
print(accuracy)

24/24 [==============================] - 1s 29ms/step - loss: 0.9927 - accuracy: 0.6301
0.9926888942718506
0.6301369667053223


In [390]:
class ReduceMeanLayer(tf.keras.layers.Layer):
  def __init__(self, axis=0, **kwargs):
    super(ReduceMeanLayer, self).__init__(**kwargs)
    self.axis = axis

  def call(self, input):
    return tf.math.reduce_mean(input, axis=self.axis)

In [391]:
saved_model_path = os.path.join(MODELDIR)

input_segment = tf.keras.layers.Input(shape=(), dtype=tf.float32, name='audio')
embedding_extraction_layer = hub.KerasLayer(MODEL_HANDLE,
                                            trainable=False, name='yamnet')
_, embeddings_output, _ = embedding_extraction_layer(input_segment)
serving_outputs = model(embeddings_output)
serving_outputs = ReduceMeanLayer(axis=0, name='classifier')(serving_outputs)
serving_model = tf.keras.Model(input_segment, serving_outputs)
serving_model.save(saved_model_path, include_optimizer=False)

INFO:tensorflow:Assets written to: /home/bhkim/github/bruit-classification/model/assets


INFO:tensorflow:Assets written to: /home/bhkim/github/bruit-classification/model/assets


In [392]:
converter = tf.lite.TFLiteConverter.from_saved_model(MODELDIR)
tflite_model = converter.convert()

with open(os.path.join(MODELDIR, 'bruit_yamnet.tflite'), 'wb') as f:
    f.write(tflite_model)

2023-06-02 10:45:47.480965: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:357] Ignored output_format.
2023-06-02 10:45:47.480988: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:360] Ignored drop_control_dependency.
2023-06-02 10:45:47.481109: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /home/bhkim/github/bruit-classification/model
2023-06-02 10:45:47.493534: I tensorflow/cc/saved_model/reader.cc:78] Reading meta graph with tags { serve }
2023-06-02 10:45:47.493558: I tensorflow/cc/saved_model/reader.cc:119] Reading SavedModel debug info (if present) from: /home/bhkim/github/bruit-classification/model
2023-06-02 10:45:47.545192: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2023-06-02 10:45:47.694305: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: /home/bhkim/github/bruit-classification/model
2023-06-02 10:45:47.788971: I tensorflow/cc/saved_mod